# 👗 Pincher — DeepFashion2 Clothing Classifier (Self-Contained Colab)
This notebook trains a lightweight **MobileNetV3** model directly on your small dataset without needing Google Drive permissions.

### ⚡ 3 Super Simple Steps:
1. In top menu: **Runtime > Change runtime type** ➔ Select **T4 GPU** ➔ **Save**.
2. On the left sidebar, click the **Folder icon** 📁 and drag & drop **`train_small.zip`** into it.
3. Click **Runtime > Run all**!

### 1️⃣ Check GPU & Extract `train_small.zip`

In [ ]:
import os, glob, shutil, json
import torch
from PIL import Image
from tqdm import tqdm

print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Using GPU:', torch.cuda.get_device_name(0))

# Check for train_small.zip in /content or Drive
if os.path.exists('/content/train_small.zip'):
    print('Found /content/train_small.zip! Extracting...')
    !unzip -q -o /content/train_small.zip -d /content/
elif os.path.exists('/content/drive/MyDrive/train_small.zip'):
    print('Found in Google Drive! Extracting...')
    !unzip -q -o /content/drive/MyDrive/train_small.zip -d /content/
else:
    print('⚠️ Please upload train_small.zip using the folder icon 📁 on the left sidebar!')

### 2️⃣ Preprocess & Crop Garments into Pincher Categories

In [ ]:
PROCESSED_DIR = '/content/pincher_dataset'
PINCHER_CLASSES = ['tops', 'bottoms', 'outerwear', 'dresses']

# Create category folders
for split in ['train', 'val']:
    for cls in PINCHER_CLASSES:
        os.makedirs(os.path.join(PROCESSED_DIR, split, cls), exist_ok=True)

CATEGORY_MAP = {
    1: 'tops',        # short sleeve top
    2: 'tops',        # long sleeve top
    3: 'outerwear',   # short sleeve outwear
    4: 'outerwear',   # long sleeve outwear
    5: 'tops',        # vest
    6: 'tops',        # sling
    7: 'bottoms',     # shorts
    8: 'bottoms',     # trousers
    9: 'bottoms',     # skirt
    10: 'dresses',    # short sleeve dress
    11: 'dresses',    # long sleeve dress
    12: 'dresses',    # vest dress
    13: 'dresses'     # sling dress
}

# Locate image and annos directories
img_dirs = glob.glob('/content/**/image', recursive=True)
anno_dirs = glob.glob('/content/**/annos', recursive=True)

if img_dirs and anno_dirs:
    img_dir = img_dirs[0]
    ann_dir = anno_dirs[0]
    print(f'Images: {img_dir}\nAnnotations: {ann_dir}')
    
    anno_files = [f for f in os.listdir(ann_dir) if f.endswith('.json')]
    print(f'Processing and cropping {len(anno_files)} clothing items...')
    
    extracted_count = 0
    for idx, filename in enumerate(tqdm(anno_files)):
        base = os.path.splitext(filename)[0]
        img_path = os.path.join(img_dir, f'{base}.jpg')
        json_path = os.path.join(ann_dir, filename)
        
        if not os.path.exists(img_path):
            continue
            
        try:
            with open(json_path, 'r') as f:
                data = json.load(f)
            img = Image.open(img_path).convert('RGB')
            
            for key, item in data.items():
                if key.startswith('item') and item.get('category_id') in CATEGORY_MAP:
                    bbox = item.get('bounding_box')
                    if bbox and (bbox[2] - bbox[0]) > 40 and (bbox[3] - bbox[1]) > 40:
                        cat = CATEGORY_MAP[item['category_id']]
                        split = 'train' if idx % 5 != 0 else 'val'
                        crop = img.crop(bbox)
                        crop.save(os.path.join(PROCESSED_DIR, split, cat, f'{base}_{key}.jpg'), quality=90)
                        extracted_count += 1
        except Exception:
            continue
            
    print(f'✅ Successfully extracted {extracted_count} clean garment crops!')
else:
    print('❌ Error: Could not locate image and annos folders. Please upload train_small.zip.')

### 3️⃣ Train Lightweight MobileNetV3 (Fast GPU Training)

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

transforms_map = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

train_ds = datasets.ImageFolder(f'{PROCESSED_DIR}/train', transforms_map['train'])
val_ds = datasets.ImageFolder(f'{PROCESSED_DIR}/val', transforms_map['val'])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

print('Categories:', train_ds.classes)
print(f'Train Samples: {len(train_ds)} | Validation Samples: {len(val_ds)}')

# MobileNetV3 Small (9 MB, ultra-fast inference)
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(train_ds.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)

num_epochs = 4
print('\n🚀 Starting Training on GPU (takes ~2 minutes)...')
for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
        train_correct += (out.argmax(1) == y).sum().item()
        
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            val_correct += (model(x).argmax(1) == y).sum().item()
            
    train_acc = train_correct / len(train_ds) * 100
    val_acc = val_correct / len(val_ds) * 100
    print(f'Epoch {epoch+1}/{num_epochs} -> Train Accuracy: {train_acc:.1f}% | Validation Accuracy: {val_acc:.1f}%')

print('\n✨ Training Complete!')

### 4️⃣ Export Model to ONNX & Download Automatically

In [ ]:
from google.colab import files

# Export to ONNX
dummy = torch.randn(1, 3, 224, 224, device=device)
torch.onnx.export(
    model,
    dummy,
    '/content/pincher_clothing_model.onnx',
    input_names=['image'],
    output_names=['category_scores'],
    dynamic_axes={'image': {0: 'batch_size'}, 'category_scores': {0: 'batch_size'}}
)

print('🎉 Model exported! Starting automatic download to your browser...')
files.download('/content/pincher_clothing_model.onnx')